In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd 
import cartopy.crs as ccrs
import cmocean
from scipy.stats import pearsonr
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from OceanDataStore import OceanDataCatalog 
from nemo_cookbook import NEMODataTree  
from matplotlib.patches import Rectangle


In [5]:
# Open model and config files

catalog = OceanDataCatalog(catalog_name='noc-model-stac')
catalog.search(collection='noc-npd-era5', standard_name='sea_surface_temperature')
ds1_annual = catalog.open_dataset(id=catalog.Items[2].id,
                          start_datetime='1990-01',
                          end_datetime='2024-12')
ds1_5day = catalog.open_dataset(id=catalog.Items[4].id,
                          start_datetime='1990-01',
                          end_datetime='2024-12')

catalog.search(collection='noc-npd-era5', item_name='domain_cfg')
config = catalog.open_dataset(id=catalog.Items[1].id)

# Merge into a data tree
datasets_annual = {'parent': {'domain': config, 'gridT': ds1_annual}}
dt_global_annual = NEMODataTree.from_datasets(datasets = datasets_annual)
datasets_5day = {'parent': {'domain': config, 'gridT': ds1_5day}}
dt_global_5day = NEMODataTree.from_datasets(datasets = datasets_5day)

# Clip to North Atlantic 
bbox = (-85.0, 0.0, 0.0, 80.0)
dt_annual = dt_global_annual.clip_grid(grid='/gridT', bbox=bbox)
dt_5day = dt_global_5day.clip_grid(grid='/gridT', bbox=bbox)

# Correct compatibility error 
dt_annual['gridT']['tmaskutil'] = dt_annual['gridT']['tmaskutil'].astype(bool)
dt_5day['gridT']['tmaskutil'] = dt_5day['gridT']['tmaskutil'].astype(bool)

# Convert to datasets
ds_annual = (dt_annual['/gridT']).dataset
ds_5day = (dt_5day['/gridT']).dataset


            * Item ID: noc-npd-era5/npd-eorca1-era5v1/gn/T1y
              Title: eORCA1 ERA5v1 NPD T1y Icechunk repository
              Description: Icechunk repository containing eORCA1 ERA5v1 NPD global ocean physics annual mean outputs defined at T-points.
              Platform: gn
              Start Date: 1976-01-01T00:00:00Z
              End Date: 2025-07-31T00:00:00Z
            

            * Item ID: noc-npd-era5/npd-eorca1-era5v1/gn/T1m
              Title: eORCA1 ERA5v1 NPD T1m Icechunk repository
              Description: Icechunk repository containing eORCA1 ERA5v1 NPD global ocean physics monthly mean outputs defined at T-points.
              Platform: gn
              Start Date: 1976-01-01T00:00:00Z
              End Date: 2025-07-31T00:00:00Z
            

            * Item ID: noc-npd-era5/npd-eorca025-era5v1/gn/T1y_3d
              Title: eORCA025 ERA5v1 NPD T1y_3d Icechunk repository
              Description: Icechunk repository containing eORCA025 ERA5v1 

In [6]:
## Compute data 

min_region = ds_5day.sel(time_counter = ds_5day['time_counter'].dt.month.isin([2, 3, 4]))
max_region = ds_5day.sel(time_counter = ds_5day['time_counter'].dt.month.isin([8, 9, 10]))
min_temp = min_region['tos_con'].groupby(min_region['time_counter'].dt.year).min(dim='time_counter')
max_temp = max_region['tos_con'].groupby(max_region['time_counter'].dt.year).max(dim='time_counter')
print('1')
amplitude = (max_temp - min_temp).compute()
print('2')
salinity = ds_annual['sos_abs'].compute()

1
2


In [8]:
salinity

<xarray.DataArray 'sos_abs' (time_counter: 35, j: 482, i: 341)> Size: 23MB
array([[[      nan, 34.26697 , 34.264153, ..., 35.34912 , 35.32821 ,
         35.30918 ],
        [      nan, 34.204773, 34.20046 , ..., 35.314888, 35.295815,
         35.27701 ],
        [      nan, 34.157356, 34.151146, ..., 35.302853, 35.287457,
         35.271465],
        ...,
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan]],

       [[      nan, 34.263077, 34.246174, ..., 35.15142 , 35.12914 ,
         35.101486],
        [      nan, 34.196728, 34.17971 , ..., 35.12395 , 35.097218,
         35.072742],
        [      nan, 34.137012, 34.11754 , ..., 35.102985, 35.07608 ,
         35.05439 ],
...
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan]],

       [[      nan, 34.017017, 34.005142, ..., 35.216213, 35.197655,
         35.1789  ],
        [      nan, 33.941578, 33.92331 , ..., 35.16202 , 35.143944,
         35.12868 ],
        [      nan, 33.875042, 33.850513, ..., 35.13272 , 35.115757,
         35.10049 ],
        ...,
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan]]], shape=(35, 482, 341), dtype=float32)
Coordinates:
  * time_counter   (time_counter) datetime64[ns] 280B 1990-07-02T12:00:00 ......
  * j              (j) int64 4kB 685 686 687 688 689 ... 1163 1164 1165 1166
  * i              (i) int64 3kB 809 810 811 812 813 ... 1146 1147 1148 1149
    time_centered  (time_counter) datetime64[ns] 280B 1990-07-02T12:00:00 ......
    gphit          (j, i) float64 1MB 0.0 0.0 0.0 0.0 ... 79.51 79.41 79.3 79.2
    glamt          (j, i) float64 1MB -85.0 -84.75 -84.5 ... 48.28 48.54 48.79
Attributes:
    standard_name:       sea_surface_salinity
    long_name:           sea_surface_absolute_salinity
    units:               g/kg
    online_operation:    average
    interval_operation:  1800 s
    interval_write:      1 yr
    cell_methods:        time: mean (interval: 1800 s)

In [13]:
amplitude.to_netcdf('amplitude_ts_0.25.nc')
salinity.to_netcdf('salinity_ts_0.25.nc')

In [14]:
## Start from here in future 

salinity = xr.open_dataset('salinity_ts_0.25.nc')
amplitude = xr.open_dataset('amplitude_ts_0.25.nc')


In [40]:
## Compute Correlation coefficients 

ny, nx = amplitude.sizes['j'], amplitude.sizes['i']
r_data = np.full((ny, nx), np.nan, dtype = np.float32)
p_data = np.full((ny, nx), np.nan, dtype = np.float32)

print('starting loop')
for y_idx in range (ny):
    for x_idx in range (nx):
        amp_ts = amplitude.isel(j=y_idx, i=x_idx)['tos_con'].values
        sal_ts = salinity.isel(j=y_idx, i=x_idx)['sos_abs'].values
        try:
            r, p = pearsonr(sal_ts, amp_ts)
            r_data[y_idx, x_idx] = r
            p_data[y_idx, x_idx] = p
        except:
            r_data[y_idx, x_idx] = np.nan
            p_data[y_idx, x_idx] = np.nan
    print(y_idx)


starting loop
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273


In [44]:
r_da = xr.DataArray(data = r_data, dims=['j', 'i'], 
        coords={ 'j': amplitude['j'],
        'i': amplitude['i'], 'gphit': (('j', 'i'), amplitude['gphit'].values),
        'glamt': (('j', 'i'), amplitude['glamt'].values)}, name='correlation',
        attrs={'description': 'Correlation coefficient between amplitude and salinity'})

pval_da = xr.DataArray(data = p_data, dims=['j', 'i'], 
        coords={ 'j': amplitude['j'],
        'i': amplitude['i'], 'gphit': (('j', 'i'), amplitude['gphit'].values),
        'glamt': (('j', 'i'), amplitude['glamt'].values)}, name='p-value',
        attrs={'description': 'P-value for correlation coefficient between amplitude and salinity'})

In [47]:
r_da

<xarray.DataArray 'correlation' (j: 482, i: 341)> Size: 657kB
array([[        nan, -0.01199781, -0.02417547, ..., -0.04293428,
        -0.04434355, -0.05025087],
       [        nan, -0.02882569, -0.03805303, ..., -0.08025693,
        -0.08084098, -0.07677577],
       [        nan, -0.00451304, -0.01464529, ..., -0.1113824 ,
        -0.09810593, -0.0855054 ],
       ...,
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan]], shape=(482, 341), dtype=float32)
Coordinates:
  * j        (j) int64 4kB 685 686 687 688 689 690 ... 1162 1163 1164 1165 1166
  * i        (i) int64 3kB 809 810 811 812 813 814 ... 1145 1146 1147 1148 1149
    gphit    (j, i) float64 1MB 0.0 0.0 0.0 0.0 0.0 ... 79.51 79.41 79.3 79.2
    glamt    (j, i) float64 1MB -85.0 -84.75 -84.5 -84.25 ... 48.28 48.54 48.79
Attributes:
    description:  Correlation coefficient between amplitude and salinity

In [ ]:
fig, ax = plt.subplots(figsize = (10,25), subplot_kw={'projection': ccrs.PlateCarree()})

im = ax.pcolormesh(r_da['glamt'], r_da['gphit'], r_da, cmap='RdBu_r', transform=ccrs.PlateCarree(), vmin = -1, vmax = 1)  
ax.set_extent([-85.0, 0.0, 0.0, 80.0], crs=ccrs.PlateCarree())
cl = ax.coastlines()
gl = ax.gridlines(draw_labels=True, linewidth=0.5, linestyle='--')
title = ax.set_title('Pearsons correlation coefficient: SST Amplitude and Mean Salinity')
box = Rectangle((-45, 40), 25, 25, linewidth=2, edgecolor='black', facecolor='none', linestyle='--', transform=ccrs.PlateCarree())
ax.add_patch(box)
ax.plot(-36, 53, 'ko', markersize=8, transform=ccrs.PlateCarree())
plt.colorbar(im)